# ML — LR, RF y XGBoost · Nested CV 5×3 con grupos · TF-IDF 20k

Evaluación con `StratifiedGroupKFold`, F1-Macro y separación estricta entre selección interna y evaluación externa. Outer CV = 5 folds; Inner CV = 3 folds.


## 1. Librerías y configuración compartida

Se evita redefinir `DATA_DIR`, `FEATURE_COLS`, `VARIANTES` o `RANDOM_STATE` en secciones distintas. Ajusta `DATA_DIR` si tu notebook no está ubicado junto al directorio `data`.

In [ ]:
import os
import json
import time
import joblib
import warnings
import html
import numpy as np
import pandas as pd
import optuna
from IPython.display import display, HTML

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_val_score
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = "../data"
OUT_DIR = os.path.join(DATA_DIR, "resultados_ml_groupcv_tfidf_xgb_20k")
MODELS_DIR = os.path.join(OUT_DIR, "modelos_finales")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

RANDOM_STATE = 42
VARIANTES = ["mx", "es", "cu"]
MODELOS_LR_RF = ["LR", "RF"]
PREPROCESAMIENTOS = {"normal": "", "stem": "_stem", "lemma": "_lemma"}
FEATURE_COLS = [
    "n_exc", "n_int", "n_may", "n_emo", "n_ris",
    "n_neg", "n_elo", "n_com", "n_pun",
]

N_OUTER = 5
N_INNER = 3
N_TRIALS_NESTED = 50
N_TRIALS_FINAL = 50
N_JOBS_GRID = -1       # Paralelismo de GridSearchCV para LR/RF
N_JOBS_XGB = -1        # Dentro de XGBoost; CV de Optuna secuencial

TFIDF_COMUN = True     # MANTENER True para comparar LR, RF y XGBoost uniformemente
EJECUTAR_TEST_OFICIAL = False  # Cambiar SOLO cuando toda la selección esté cerrada
OOF_REGISTROS = []        # Predicciones independientes del Outer Test (sin datos oficiales de test)

import sklearn
import xgboost
VERSIONES = {
    "numpy": np.__version__, "pandas": pd.__version__,
    "sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
    "optuna": optuna.__version__,
}
print("Versiones:", VERSIONES)

print("Algoritmos: LR, RF y XGBoost")
print(f"StratifiedGroup Nested CV: {N_OUTER} outer × {N_INNER} inner; XGBoost {N_TRIALS_NESTED} trials/outer")
print(f"TF-IDF común: {TFIDF_COMUN} | Test oficial habilitado: {EJECUTAR_TEST_OFICIAL}")

# ------------------------------------------------------------
# Monitor compacto para la parte larga de LR/RF.
# Una sola salida se actualiza; GridSearchCV permanece con verbose=0.
# ------------------------------------------------------------
class MonitorML:
    def __init__(self, total_bloques):
        self.total = int(total_bloques)
        self.completados = 0
        self.inicio = time.monotonic()
        self.etapa = "Inicializando LR/RF"
        self.handle = display(HTML("<b>ML — preparando GridSearch...</b>"), display_id=True)
        self.mostrar(forzar=True)

    def mostrar(self, etapa=None, forzar=False):
        if etapa is not None:
            self.etapa = str(etapa)
        minutos = (time.monotonic() - self.inicio) / 60
        faltan = max(0, self.total - self.completados)
        eta = (
            f"~{minutos / self.completados * faltan:.0f} min"
            if self.completados >= 2 else "calculando..."
        )
        panel = (
            '<div style="font-family:system-ui;padding:12px;border:1px solid #777;'
            'border-radius:8px;max-width:800px">'
            '<b>ML — seguimiento en vivo</b><br>'
            f'Etapa: <b>{html.escape(self.etapa)}</b><br>'
            f'Búsquedas GridSearch completadas: {self.completados}/{self.total} '
            f'({100*self.completados/max(1,self.total):.1f}%)<br>'
            f'<progress value="{self.completados}" max="{max(1,self.total)}" '
            'style="width:100%"></progress><br>'
            f'Tiempo: {minutos:.1f} min | ETA orientativo: {eta}'
            '</div>'
        )
        self.handle.update(HTML(panel))

    def iniciar(self, etapa):
        self.mostrar(etapa=etapa, forzar=True)

    def completar(self, etapa):
        self.completados += 1
        self.mostrar(etapa=etapa, forzar=True)


## 2. Integridad de los CSV y validación por grupos (sin entrenamiento)

La clave de agrupación no usa etiquetas. Si un texto original, normalizado, stem o lemma coincide con otro, **todos** los registros conectados quedan en el mismo grupo. `StratifiedGroupKFold` intenta preservar el balance de clases sin separar un grupo entre train/validación. Las etiquetas contradictorias se mantienen intactas y se reportan para revisión manual.

La comprobación anticipada de solapamiento con el test **solo abre texto e IDs**; no lee `IS_IRONIC`. Las coincidencias train–test normalizadas ya observadas (ES y CU) deben informarse como limitación; esta corrección arregla los folds internos/externos de desarrollo, no reescribe el test oficial.


In [ ]:
# ============================================================
# 2. CARGA, INTEGRIDAD Y GRUPOS SIN FUGA ENTRE FOLDS
# ============================================================
import hashlib
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

GRUPOS_TRAIN = {}        # Variante -> np.array; solo contiene información del TRAIN
SOLAPE_TEST_SIN_ETIQUETAS = {}  # máscaras, calculadas sin IS_IRONIC, jamás para optimizar
AUDITORIA_GRUPOS = []
AUDITORIA_CONFLICTOS = []
AUDITORIA_PARTICIONES = []


def cargar_split(variante, prep, split="train"):
    if variante not in VARIANTES or prep not in PREPROCESAMIENTOS:
        raise ValueError(f"Variante/preprocessing no reconocido: {variante}/{prep}")
    ruta = os.path.join(DATA_DIR, f"{split}_clean{PREPROCESAMIENTOS[prep]}_{variante}.csv")
    if not os.path.isfile(ruta):
        raise FileNotFoundError(f"Falta el CSV: {ruta}")
    # Preservar los IDs originales: no forzarles una conversión numérica.
    return pd.read_csv(ruta, dtype={"ID": "string"}).reset_index(drop=True)


def preparar_xy(df):
    columnas = ["MESSAGE_CLEAN"] + FEATURE_COLS
    faltantes = [c for c in columnas + ["IS_IRONIC"] if c not in df.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas: {faltantes}")
    X = df[columnas].copy()
    y = df["IS_IRONIC"].astype(int).reset_index(drop=True)
    X["MESSAGE_CLEAN"] = X["MESSAGE_CLEAN"].fillna("").astype(str)
    X[FEATURE_COLS] = X[FEATURE_COLS].fillna(0)
    return X.reset_index(drop=True), y


def texto_clave(serie):
    # Misma normalización de COMPARACIÓN que la auditoría previa; no modifica el texto del modelo.
    return serie.fillna("").astype(str).str.casefold().str.strip()


def construir_grupos(variante, dfs_por_prep, y):
    """Componentes conexos: agrupa un mismo texto tras normal, stem O lemma.
    No deduplica, no usa las etiquetas para construir grupos y conserva rasgos originales.
    Añade coincidencias de MESSAGE original exacto, si está disponible.
    """
    n = len(y)
    parent = list(range(n))
    size = [1] * n

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if size[ra] < size[rb]:
            ra, rb = rb, ra
        parent[rb] = ra
        size[ra] += size[rb]

    for prep, df in dfs_por_prep.items():
        primero = {}
        for i, clave in enumerate(texto_clave(df["MESSAGE_CLEAN"])):
            if not clave:
                raise ValueError(f"{variante}/{prep}: texto limpio vacío en fila {i}; revisar antes de agrupar")
            if clave in primero:
                union(primero[clave], i)
            else:
                primero[clave] = i
    # Si dos textos originales son idénticos, deben permanecer en el mismo grupo.
    df_ref = dfs_por_prep["normal"]
    if "MESSAGE" in df_ref:
        primero_raw = {}
        for i, clave in enumerate(texto_clave(df_ref["MESSAGE"])):
            if not clave:
                continue
            if clave in primero_raw:
                union(primero_raw[clave], i)
            else:
                primero_raw[clave] = i

    miembros = {}
    for i in range(n):
        miembros.setdefault(find(i), []).append(i)
    # Id de grupo basado en el menor índice original; determinista entre ambos notebooks.
    ids = {ra: min(idx) for ra, idx in miembros.items()}
    grupos = np.array([ids[find(i)] for i in range(n)], dtype=int)
    duplicados = {ids[ra]: idx for ra, idx in miembros.items() if len(idx) > 1}
    conflictivos = [(g, idx) for g, idx in duplicados.items() if y.iloc[idx].nunique() > 1]
    if conflictivos:
        warnings.warn(
            f"{variante.upper()}: {len(conflictivos)} grupo(s) con etiquetas contradictorias. "
            "Se conservan las etiquetas originales y se mantienen juntos todos sus registros. "
            "Revisar manualmente antes de interpretar resultados.", stacklevel=2)
    for g, idx in conflictivos:
        for i in idx:
            AUDITORIA_CONFLICTOS.append({
                "variante": variante, "grupo": int(g), "fila_0base": int(i),
                "id_original": str(df_ref.iloc[i]["ID"]) if "ID" in df_ref else "",
                "etiqueta": int(y.iloc[i]),
                "texto_original": str(df_ref.iloc[i]["MESSAGE"]) if "MESSAGE" in df_ref else "",
                "texto_normal": str(df_ref.iloc[i]["MESSAGE_CLEAN"]),
            })
    AUDITORIA_GRUPOS.append({
        "variante": variante, "n_train": n, "n_grupos": len(miembros),
        "n_grupos_multiples": len(duplicados),
        "n_filas_en_grupos_multiples": sum(map(len, duplicados.values())),
        "tamano_grupo_max": max(map(len, miembros.values())),
        "n_grupos_etiquetas_contradictorias": len(conflictivos),
        "faltantes_vs_2400": 2400-n,
        "criterio": "union_texto_normal_stem_lemma_y_original_exacto",
    })
    if n != 2400:
        warnings.warn(
            f"{variante.upper()}: {n} registros en train (faltan {2400-n} frente a la partición esperada). "
            "No se inventan ni recuperan filas; verificar contra el corpus original.", stacklevel=2)
    return grupos


def cargar_todos_preps(variante, split="train"):
    dfs = {p: cargar_split(variante, p, split=split) for p in PREPROCESAMIENTOS}
    ref = dfs["normal"]
    X_ref, y_ref = preparar_xy(ref)
    datos = {"normal": X_ref}
    for prep, df in dfs.items():
        if prep == "normal":
            continue
        X, y = preparar_xy(df)
        if len(df) != len(ref) or not y.equals(y_ref):
            raise ValueError(f"{variante}/{prep}: diferente número de filas o etiquetas")
        for c in ["ID", "MESSAGE"]:
            if c in ref.columns and c not in df.columns:
                raise ValueError(f"{variante}/{prep}: falta columna de alineación {c}")
            if c in ref.columns and not ref[c].astype(str).equals(df[c].astype(str)):
                raise ValueError(f"{variante}/{prep}: columna {c} desalineada")
        if not X[FEATURE_COLS].equals(X_ref[FEATURE_COLS]):
            raise ValueError(f"{variante}/{prep}: rasgos lingüísticos no alineados")
        datos[prep] = X
    if split == "train" and variante not in GRUPOS_TRAIN:
        GRUPOS_TRAIN[variante] = construir_grupos(variante, dfs, y_ref)
    return datos, y_ref


def grupos_train(variante):
    if variante not in GRUPOS_TRAIN:
        cargar_todos_preps(variante, split="train")
    return GRUPOS_TRAIN[variante]


def verificar_particion(grupos, a, b, variante, nivel, outer, inner=None):
    solapamiento = set(grupos[a]) & set(grupos[b])
    if solapamiento:
        raise AssertionError(f"FUGA: {variante}/{nivel}/{outer}/{inner} grupos compartidos: {solapamiento}")
    AUDITORIA_PARTICIONES.append({
        "variante": variante, "nivel": nivel, "outer_fold": outer,
        "inner_fold": inner, "n_train": len(a), "n_validacion": len(b),
        "n_grupos_compartidos": 0,
    })


def crear_splits_outer(y, grupos, variante="?"):
    cv = StratifiedGroupKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_STATE)
    splits = list(cv.split(np.zeros(len(y)), y, groups=grupos))
    for fold, (a, b) in enumerate(splits, 1):
        verificar_particion(grupos, a, b, variante, "outer", fold)
    return splits


def crear_splits_inner(y, grupos, outer_fold, variante="?"):
    cv = StratifiedGroupKFold(n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE+outer_fold)
    splits = list(cv.split(np.zeros(len(y)), y, groups=grupos))
    for fold, (a, b) in enumerate(splits, 1):
        verificar_particion(grupos, a, b, variante, "inner", outer_fold, fold)
    return splits


def crear_splits_final(y, grupos, variante="?"):
    cv = StratifiedGroupKFold(n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE)
    splits = list(cv.split(np.zeros(len(y)), y, groups=grupos))
    for fold, (a, b) in enumerate(splits, 1):
        verificar_particion(grupos, a, b, variante, "final", 0, fold)
    return splits


def verificar_integridad_test_sin_etiquetas(variante):
    """Verifica los tres test CSV SIN leer IS_IRONIC ni calcular métricas.

    Los solapes se buscan en cualquier versión normal/stem/lemma, además del original.
    Se guarda máscara para mostrar sensibilidad al final, NO para elegir modelos.
    """
    rutas = {p: os.path.join(DATA_DIR, f"test_clean{PREPROCESAMIENTOS[p]}_{variante}.csv")
             for p in PREPROCESAMIENTOS}
    faltantes = [p for p, ruta in rutas.items() if not os.path.isfile(ruta)]
    if faltantes:
        return {"variante": variante, "estado": "faltan versiones de test: " + str(faltantes),
                "solape_cualquier_rep": None}
    columnas = {"ID", "MESSAGE", "MESSAGE_CLEAN"}
    tests = {p: pd.read_csv(ruta, usecols=lambda c: c in columnas, dtype="string").fillna("")
             for p, ruta in rutas.items()}
    train = {p: cargar_split(variante, p, split="train") for p in PREPROCESAMIENTOS}
    referencia = tests["normal"]
    solapados = np.zeros(len(referencia), dtype=bool)
    por_prep = {}
    for prep in PREPROCESAMIENTOS:
        dft = tests[prep]
        if len(dft) != len(referencia):
            raise ValueError(f"{variante}/{prep}: test desalineado en número de filas")
        for c in ["ID", "MESSAGE"]:
            if c in referencia.columns and c not in dft.columns:
                raise ValueError(f"{variante}/{prep}: test sin columna {c}")
            if c in referencia.columns and not dft[c].astype(str).equals(referencia[c].astype(str)):
                raise ValueError(f"{variante}/{prep}: IDs o originales de test desalineados")
        claves_train = set(texto_clave(train[prep]["MESSAGE_CLEAN"])) - {""}
        coincidencias = texto_clave(dft["MESSAGE_CLEAN"]).isin(claves_train).to_numpy()
        por_prep[prep] = int(coincidencias.sum())
        solapados |= coincidencias
    solape_original = None
    solape_id = None
    if "MESSAGE" in referencia and "MESSAGE" in train["normal"]:
        claves_originales = set(texto_clave(train["normal"]["MESSAGE"])) - {""}
        coincidencias = texto_clave(referencia["MESSAGE"]).isin(claves_originales).to_numpy()
        solape_original = int(coincidencias.sum())
        solapados |= coincidencias
    if "ID" in referencia and "ID" in train["normal"]:
        ids_train = set(train["normal"]["ID"].fillna("").astype(str)) - {""}
        coincidencias = referencia["ID"].fillna("").astype(str).isin(ids_train).to_numpy()
        solape_id = int(coincidencias.sum())
        solapados |= coincidencias
    SOLAPE_TEST_SIN_ETIQUETAS[variante] = solapados
    return {"variante": variante, "estado": "sin leer etiquetas", "n_test": len(referencia),
            "solape_normal": por_prep["normal"], "solape_stem": por_prep["stem"],
            "solape_lemma": por_prep["lemma"],
            "solape_original": solape_original, "solape_id": solape_id,
            "solape_cualquier_rep": int(solapados.sum())}


# Auditoría previa obligatoria ANTES de entrenar: valida los mismos grupos en AMBOS notebooks.
for v in VARIANTES:
    _, etiquetas = cargar_todos_preps(v, split="train")
    gv = grupos_train(v)
    for o, (a, b) in enumerate(crear_splits_outer(etiquetas, gv, v), 1):
        crear_splits_inner(etiquetas.iloc[a].reset_index(drop=True), gv[a], o, v)
    crear_splits_final(etiquetas, gv, v)

df_auditoria_grupos = pd.DataFrame(AUDITORIA_GRUPOS)
df_auditoria_particiones = pd.DataFrame(AUDITORIA_PARTICIONES).drop_duplicates()
df_auditoria_conflictos = pd.DataFrame(AUDITORIA_CONFLICTOS)
df_integridad_test = pd.DataFrame([verificar_integridad_test_sin_etiquetas(v) for v in VARIANTES])
print("AUDITORÍA PREVIA: mismos grupos en los dos experimentos")
display(df_auditoria_grupos)
print("Solapamientos entre folds (todos deben ser 0):")
display(df_auditoria_particiones.groupby(["variante", "nivel"], as_index=False)["n_grupos_compartidos"].max())
print("Comprobación de TRAIN/TEST sin etiquetas del test:")
display(df_integridad_test)
for v in VARIANTES:
    n = df_integridad_test.loc[df_integridad_test.variante == v, "solape_cualquier_rep"].iloc[0]
    if n:
        warnings.warn(f"{v.upper()}: {n} textos de TEST coinciden con TRAIN tras normalización. "
                      "Los folds agrupados NO corrigen ese solapamiento del test oficial; "
                      "reportar sensibilidad en la evaluación final, sin seleccionar con test.", stacklevel=2)

pd.DataFrame(AUDITORIA_GRUPOS).to_csv(os.path.join(OUT_DIR,"auditoria_grupos.csv"), index=False)
pd.DataFrame(AUDITORIA_PARTICIONES).drop_duplicates().to_csv(os.path.join(OUT_DIR,"auditoria_folds.csv"), index=False)
df_auditoria_conflictos.to_csv(os.path.join(OUT_DIR,"auditoria_etiquetas_contradictorias.csv"), index=False)
df_integridad_test.to_csv(os.path.join(OUT_DIR,"auditoria_test_sin_etiquetas.csv"), index=False)
print("Auditoría guardada; si aparece algún solapamiento entre folds, el notebook se detiene.")

## 3. TF-IDF y características lingüísticas

Con `TFIDF_COMUN=True`, los tres modelos usan exactamente la configuración de TF-IDF de tu notebook XGBoost: palabras, 1–2 gramas, 20 000 términos como máximo, `min_df=2`, `max_df=.98`, `sublinear_tf=True`, `lowercase=False` y 9 características lingüísticas. Con `False`, LR/RF recuperan su TF-IDF original (stopwords españolas, hasta 10 000 características y 1–1 o 1–2 gramas elegidos en su grid) y XGBoost conserva el suyo. La opción `False` no aísla el efecto del algoritmo en la comparación.

In [ ]:
def crear_preprocesador_xgb():
    tfidf = TfidfVectorizer(
        analyzer="word", ngram_range=(1, 2), min_df=2,
        max_df=0.98, max_features=20000, lowercase=False,
        sublinear_tf=True, dtype=np.float32,
    )
    return ColumnTransformer(
        transformers=[
            ("tfidf", tfidf, "MESSAGE_CLEAN"),
            ("linguisticas", "passthrough", FEATURE_COLS),
        ],
        remainder="drop",
    )


def crear_preprocesador_lr_rf():
    if TFIDF_COMUN:
        return crear_preprocesador_xgb()
    # Reproducción opcional de la representación del notebook LR/RF original.
    from nltk.corpus import stopwords
    stop_words = stopwords.words("spanish")  # Requiere el corpus NLTK instalado.
    tfidf = TfidfVectorizer(stop_words=stop_words, max_features=10000)
    return ColumnTransformer(
        [("tfidf", tfidf, "MESSAGE_CLEAN"),
         ("linguisticas", "passthrough", FEATURE_COLS)]
    )

print("Representaciones definidas.")

## 4. LR y RF — GridSearchCV con preprocessing dentro del Inner CV

**Corrección respecto del LR/RF original**: ya no se ejecuta un Nested CV independiente para cada preprocesamiento y se escoge después el más alto mirando los Outer scores. En cada Outer Fold se evalúan `normal`, `stem` y `lemma` con GridSearchCV sobre los **mismos cinco Inner folds**, y solo su puntuación interna decide qué combinación se lleva al Outer Test.

Con TF-IDF común, la representación es fija; se buscan únicamente hiperparámetros del clasificador. Con representación original, el grid también decide el rango de n-gramas.

In [ ]:
def crear_pipeline_lr_rf(modelo):
    if modelo == "LR":
        clf = LogisticRegression(
            max_iter=10000, random_state=RANDOM_STATE, class_weight="balanced"
        )
    elif modelo == "RF":
        clf = RandomForestClassifier(
            random_state=RANDOM_STATE, class_weight="balanced", n_jobs=1
        )
    else:
        raise ValueError(f"Modelo LR/RF no reconocido: {modelo}")
    return Pipeline(steps=[("features", crear_preprocesador_lr_rf()), ("clf", clf)])


def grid_lr_rf(modelo):
    if modelo == "LR":
        grid = {"clf__C": [0.1, 1.0, 10], "clf__solver": ["liblinear"]}
    elif modelo == "RF":
        grid = {
            "clf__n_estimators": [100, 200, 300],
            "clf__max_depth": [None, 10, 20],
            "clf__min_samples_split": [2, 5],
        }
    else:
        raise ValueError(modelo)
    if not TFIDF_COMUN:
        grid["features__tfidf__ngram_range"] = [(1, 1), (1, 2)]
    return grid


def buscar_lr_rf_train(X_por_prep, y, modelo, inner_splits, monitor=None, variante=None, outer_fold=None):
    """Selecciona preprocessing + hiperparámetros EXCLUSIVAMENTE por puntuación Inner CV."""
    mejor = None
    candidatos = []
    for prep, X in X_por_prep.items():
        etiqueta = (
            f"[{str(variante).upper()}][{modelo}][Outer {outer_fold}/{N_OUTER}] "
            f"GridSearch | prep={prep}"
        )
        if monitor is not None:
            monitor.iniciar(etiqueta)
        search = GridSearchCV(
            estimator=crear_pipeline_lr_rf(modelo),
            param_grid=grid_lr_rf(modelo),
            cv=inner_splits,
            scoring="f1_macro",
            n_jobs=N_JOBS_GRID,
            refit=True,
            error_score="raise",
        )
        search.fit(X, y)
        if monitor is not None:
            monitor.completar(etiqueta + " | completado")
        scores_folds = [
            search.cv_results_[f"split{i}_test_score"][search.best_index_]
            for i in range(len(inner_splits))
        ]
        desviacion = float(np.std(scores_folds, ddof=1))
        candidato = {
            "prep": prep,
            "inner_f1": float(search.best_score_),
            "inner_std": desviacion,
            "params": search.best_params_,
            "estimator": search.best_estimator_,
        }
        candidatos.append({k: v for k, v in candidato.items() if k != "estimator"})
        if (mejor is None or candidato["inner_f1"] > mejor["inner_f1"]
            or (np.isclose(candidato["inner_f1"], mejor["inner_f1"])
                and candidato["inner_std"] < mejor["inner_std"])):
            mejor = candidato
    return mejor, candidatos


def nested_cv_lr_rf_variante(variante, modelo, monitor=None):
    X_por_prep, y = cargar_todos_preps(variante, split="train")
    detalle, candidatos_inner = [], []
    grupos = grupos_train(variante)
    for outer_fold, (i_train, i_test) in enumerate(crear_splits_outer(y, grupos, variante), start=1):
        y_train, y_test = y.iloc[i_train].reset_index(drop=True), y.iloc[i_test].reset_index(drop=True)
        X_train = {p: X.iloc[i_train].reset_index(drop=True) for p, X in X_por_prep.items()}
        X_outer = {p: X.iloc[i_test].reset_index(drop=True) for p, X in X_por_prep.items()}
        mejor, candidatos = buscar_lr_rf_train(
            X_train, y_train, modelo,
            crear_splits_inner(y_train, grupos[i_train], outer_fold, variante),
            monitor=monitor, variante=variante, outer_fold=outer_fold,
        )
        for c in candidatos:
            candidatos_inner.append({
                "variante": variante, "modelo": modelo, "outer_fold": outer_fold,
                "preprocessing": c["prep"], "inner_f1_macro": c["inner_f1"],
                "inner_f1_std": c["inner_std"], "best_params": c["params"],
            })
        prep = mejor["prep"]
        # search.best_estimator_ ya es un refit nuevo sobre TODO este outer_train.
        estimador = mejor["estimator"]
        pred_train = estimador.predict(X_train[prep])
        pred_outer = estimador.predict(X_outer[prep])
        train_f1 = f1_score(y_train, pred_train, average="macro")
        outer_f1 = f1_score(y_test, pred_outer, average="macro")
        # Cada registro figura exactamente una vez como Outer Test en una CV 5-fold.
        OOF_REGISTROS.append(pd.DataFrame({
            "variante": variante, "modelo": modelo, "outer_fold": outer_fold,
            "idx_train_oficial": i_test, "y_true": y_test.to_numpy(),
            "y_pred": np.asarray(pred_outer),
        }))
        f1_clase = f1_score(y_test, pred_outer, labels=[0, 1], average=None, zero_division=0)
        detalle.append({
            "variante": variante, "modelo": modelo, "outer_fold": outer_fold,
            "outer_f1_clase_0": float(f1_clase[0]), "outer_f1_clase_1": float(f1_clase[1]),
            "best_preprocessing": prep, "train_f1_macro": float(train_f1),
            "inner_best_f1": mejor["inner_f1"], "inner_best_std": mejor["inner_std"],
            "outer_f1_macro": float(outer_f1),
            "generalization_gap": float(train_f1 - outer_f1),
            "best_params": mejor["params"],
        })
        print(
            f"[{variante.upper()}][{modelo}][Outer {outer_fold}/{N_OUTER}] FINALIZADO | "
            f"prep={prep} | F1_inner={mejor['inner_f1']:.4f}±{mejor['inner_std']:.4f} | "
            f"F1_train={train_f1:.4f} | F1_outer={outer_f1:.4f} | gap={train_f1 - outer_f1:.4f}"
        )
    return pd.DataFrame(detalle), pd.DataFrame(candidatos_inner)

### 4.1 Ejecutar Nested CV de LR y RF (solo train)

Esta celda es computacionalmente costosa: evalúa las combinaciones de cada grid sobre los 5 Inner folds y las 3 alternativas de preprocessing en cada uno de los 5 Outer folds.

In [ ]:
total_bloques_lr_rf = len(VARIANTES) * len(MODELOS_LR_RF) * N_OUTER * len(PREPROCESAMIENTOS)
monitor_lr_rf = MonitorML(total_bloques=total_bloques_lr_rf)

resultados_nested_lr_rf = {}
registros_lr_rf = []
candidatos_inner_lr_rf = []

for variante in VARIANTES:
    for modelo in MODELOS_LR_RF:
        detalle, candidatos = nested_cv_lr_rf_variante(variante, modelo, monitor=monitor_lr_rf)
        resultados_nested_lr_rf[(variante, modelo)] = detalle
        candidatos_inner_lr_rf.append(candidatos)
        registros_lr_rf.append({
            "variante": variante,
            "modelo": modelo,
            "f1_train_mean": detalle["train_f1_macro"].mean(),
            "f1_nested_mean": detalle["outer_f1_macro"].mean(),
            "f1_nested_std": detalle["outer_f1_macro"].std(ddof=1),
            "gap_mean": detalle["generalization_gap"].mean(),
        })

df_nested_lr_rf = pd.DataFrame(registros_lr_rf)
df_candidatos_inner_lr_rf = pd.concat(candidatos_inner_lr_rf, ignore_index=True)
monitor_lr_rf.mostrar("LR/RF completado", forzar=True)
print("\n[ML][LR/RF] RESUMEN NESTED CV")
display(df_nested_lr_rf.round(4))

## 5. XGBoost — Optuna TPE, 50 trials y selección interna de preprocessing

Se conserva el espacio de búsqueda regularizado de tu notebook XGBoost, con 50 trials **en total por Outer Fold**, no 50 por preprocessing. Como mejora mínima, se reservan las tres primeras propuestas para probar una vez cada preprocessing (`normal`, `stem`, `lemma`); los 47 trials restantes quedan guiados por el sampler. Se registra cuántos trials completados hubo por preprocessing, ya que TPE no los distribuye necesariamente de modo uniforme.

**Diferencia legítima entre algoritmos:** LR y RF recorren sus rejillas exhaustivamente con GridSearchCV; XGBoost explora su propio espacio mediante TPE. Esto no significa que un algoritmo tenga necesariamente ventaja computacional o estadística; el presupuesto y espacio se reportan por separado.

In [ ]:
# ============================================================
# 4. ESPACIO DE BÚSQUEDA PARA OPTUNA
# ============================================================

def sugerir_params_xgb(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 150, 700, step=50),
        "max_depth": trial.suggest_int("max_depth", 2, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 2, 15),
        "subsample": trial.suggest_float("subsample", 0.55, 0.90),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.50, 0.90),
        "gamma": trial.suggest_float("gamma", 0.5, 8.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 20.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 2.5),
    }


def crear_xgboost(params, random_state=RANDOM_STATE):
    return XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",
        eval_metric="logloss",
        random_state=random_state,
        n_jobs=N_JOBS_XGB,
        **params,
    )


def crear_pipeline(params, random_state=RANDOM_STATE):
    return Pipeline(
        steps=[
            ("features", crear_preprocesador()),
            ("xgb", crear_xgboost(params, random_state=random_state)),
        ]
    )

In [ ]:
# Alias utilizado por la función crear_pipeline del notebook original de XGBoost.
# Se mantiene separado para no alterar el preprocesador de LR/RF si TFIDF_COMUN=False.
crear_preprocesador = crear_preprocesador_xgb

In [ ]:
def nested_cv_variante(variante, n_trials=N_TRIALS_NESTED):
    X_por_prep, y = cargar_todos_preps(variante, split="train")

    grupos = grupos_train(variante)

    resultados_outer = []
    studies_outer = {}
    X_ref = X_por_prep["normal"]

    for outer_fold, (idx_train, idx_test) in enumerate(
        crear_splits_outer(y, grupos, variante), start=1
    ):
        print(f"\n[{variante.upper()}][XGBoost][Outer {outer_fold}/{N_OUTER}] Optuna TPE")

        y_outer_train = y.iloc[idx_train].reset_index(drop=True)
        y_outer_test = y.iloc[idx_test].reset_index(drop=True)

        X_outer_train = {
            prep: X.iloc[idx_train].reset_index(drop=True)
            for prep, X in X_por_prep.items()
        }
        X_outer_test = {
            prep: X.iloc[idx_test].reset_index(drop=True)
            for prep, X in X_por_prep.items()
        }

        inner_splits = crear_splits_inner(y_outer_train, grupos[idx_train], outer_fold, variante)

        def objective(trial):
            prep = trial.suggest_categorical(
                "preprocessing", list(PREPROCESAMIENTOS.keys())
            )
            params = sugerir_params_xgb(trial)
            pipeline = crear_pipeline(
                params, random_state=RANDOM_STATE + outer_fold
            )

            scores = cross_val_score(
                estimator=pipeline,
                X=X_outer_train[prep],
                y=y_outer_train,
                cv=inner_splits,
                scoring="f1_macro",
                n_jobs=1,
            )

            trial.set_user_attr("inner_f1_std", float(scores.std(ddof=1)))
            return float(scores.mean())

        sampler = optuna.samplers.TPESampler(
            seed=RANDOM_STATE + outer_fold
        )
        study = optuna.create_study(
            direction="maximize", sampler=sampler
        )
        # Cada preprocessing se evalúa al menos una vez; los otros trials los decide TPE.
        if n_trials >= len(PREPROCESAMIENTOS):
            for prep_inicial in PREPROCESAMIENTOS:
                study.enqueue_trial({"preprocessing": prep_inicial})
        study.optimize(
            objective,
            n_trials=n_trials,
            show_progress_bar=True,
        )
        studies_outer[outer_fold] = study

        best_all = study.best_params.copy()
        best_prep = best_all.pop("preprocessing")
        best_params = best_all

        modelo_outer = crear_pipeline(
            best_params,
            random_state=RANDOM_STATE + outer_fold,
        )
        modelo_outer.fit(
            X_outer_train[best_prep], y_outer_train
        )

        pred_train = modelo_outer.predict(X_outer_train[best_prep])
        pred_outer = modelo_outer.predict(X_outer_test[best_prep])

        f1_train = f1_score(y_outer_train, pred_train, average="macro")
        f1_outer = f1_score(y_outer_test, pred_outer, average="macro")
        gap = f1_train - f1_outer

        OOF_REGISTROS.append(pd.DataFrame({
            "variante": variante, "modelo": "XGBoost", "outer_fold": outer_fold,
            "idx_train_oficial": idx_test, "y_true": y_outer_test.to_numpy(),
            "y_pred": np.asarray(pred_outer),
        }))
        f1_clase = f1_score(y_outer_test, pred_outer, labels=[0, 1], average=None, zero_division=0)
        resultados_outer.append({
            "outer_f1_clase_0": float(f1_clase[0]),
            "outer_f1_clase_1": float(f1_clase[1]),
            "variante": variante,
            "outer_fold": outer_fold,
            "best_preprocessing": best_prep,
            "train_f1_macro": f1_train,
            "inner_best_f1": study.best_value,
            "inner_best_std": study.best_trial.user_attrs.get(
                "inner_f1_std", np.nan
            ),
            "outer_f1_macro": f1_outer,
            "generalization_gap": gap,
            "best_params": best_params,
        })

        print(
            f"[{variante.upper()}][XGBoost][Outer {outer_fold}/{N_OUTER}] FINALIZADO | "
            f"prep={best_prep} | "
            f"F1_inner={study.best_value:.4f}±{study.best_trial.user_attrs.get('inner_f1_std', np.nan):.4f} | "
            f"F1_train={f1_train:.4f} | F1_outer={f1_outer:.4f} | gap={gap:.4f}"
        )

    df_outer = pd.DataFrame(resultados_outer)
    resumen = {
        "variante": variante,
        "f1_train_mean": df_outer["train_f1_macro"].mean(),
        "f1_nested_mean": df_outer["outer_f1_macro"].mean(),
        "f1_nested_std": df_outer["outer_f1_macro"].std(ddof=1),
        "f1_nested_min": df_outer["outer_f1_macro"].min(),
        "f1_nested_max": df_outer["outer_f1_macro"].max(),
        "gap_mean": df_outer["generalization_gap"].mean(),
    }
    return df_outer, resumen, studies_outer

### 5.1 Ejecutar Nested CV de XGBoost (solo train)

**Costo aproximado:** 3 variantes × 5 Outer × 50 trials × 5 Inner = 3 750 ajustes de XGBoost, sin contar refits. Se realizan sobre el train oficial; esta sección no carga el test.

In [ ]:
# ============================================================
# 6. EJECUTAR NESTED CV PARA LAS 3 VARIANTES
# ============================================================

nested_detalle = {}
nested_resumen = []
nested_studies = {}

for variante in VARIANTES:
    detalle, resumen, studies = nested_cv_variante(
        variante, n_trials=N_TRIALS_NESTED
    )
    nested_detalle[variante] = detalle
    nested_resumen.append(resumen)
    nested_studies[variante] = studies


df_nested_resumen = pd.DataFrame(nested_resumen)

print("\nRESULTADOS NESTED CV 5×3")
display(
    df_nested_resumen.style.format({
        "f1_train_mean": "{:.4f}",
        "f1_nested_mean": "{:.4f}",
        "f1_nested_std": "{:.4f}",
        "f1_nested_min": "{:.4f}",
        "f1_nested_max": "{:.4f}",
        "gap_mean": "{:.4f}",
    })
)

In [ ]:
# Detalle de los 5 Outer folds por variante

for variante in VARIANTES:
    print(f"\nDETALLE OUTER — {variante.upper()}")
    display(
        nested_detalle[variante][[
            "outer_fold",
            "best_preprocessing",
            "train_f1_macro",
            "inner_best_f1",
            "inner_best_std",
            "outer_f1_macro",
            "generalization_gap",
        ]].style.format({
            "train_f1_macro": "{:.4f}",
            "inner_best_f1": "{:.4f}",
            "inner_best_std": "{:.4f}",
            "outer_f1_macro": "{:.4f}",
            "generalization_gap": "{:.4f}",
        })
    )

    print("\nMejor combinación por Outer Fold:")
    for _, row in nested_detalle[variante].iterrows():
        print(
            f"  Fold {int(row['outer_fold'])}: "
            f"prep={row['best_preprocessing']} | "
            f"params={row['best_params']}"
        )

    print("\nFrecuencia de preprocessing ganador:")
    print(nested_detalle[variante]["best_preprocessing"].value_counts())

## 6. Comparación conjunta y diagnóstico — sin usar test oficial

Los tres algoritmos comparten exactamente los Outer folds, Inner folds, representación TF-IDF, nueve características y F1-Macro. Las predicciones OOF permiten calcular F1 de cada clase usando exclusivamente los 2400 registros de desarrollo. **El F1 OOF agregado no es igual, en general, al promedio de los cinco F1 externos:** son dos resúmenes complementarios.

In [ ]:
df_nested_xgb = df_nested_resumen.copy()
df_nested_xgb["modelo"] = "XGBoost"
columnas = ["variante", "modelo", "f1_train_mean", "f1_nested_mean", "f1_nested_std", "gap_mean"]
df_nested_comparativo = pd.concat(
    [df_nested_lr_rf[columnas], df_nested_xgb[columnas]],
    ignore_index=True,
).sort_values(["variante", "modelo"]).reset_index(drop=True)
print("COMPARACIÓN DE LR, RF Y XGBOOST — SOLO TRAIN OFICIAL")
display(df_nested_comparativo.style.format({
    "f1_train_mean": "{:.4f}", "f1_nested_mean": "{:.4f}",
    "f1_nested_std": "{:.4f}", "gap_mean": "{:.4f}",
}))

# Detalle uniforme para los tres modelos.
detalles = [df for df in resultados_nested_lr_rf.values()]
for variante in VARIANTES:
    d = nested_detalle[variante].copy()
    d["modelo"] = "XGBoost"
    detalles.append(d)
df_outer_comparativo = pd.concat(detalles, ignore_index=True)
print("\nDETALLE POR OUTER FOLD")
display(df_outer_comparativo[[
    "variante", "modelo", "outer_fold", "best_preprocessing",
    "train_f1_macro", "inner_best_f1", "inner_best_std",
    "outer_f1_macro", "generalization_gap", "best_params",
]].sort_values(["variante", "modelo", "outer_fold"]))

print("\nMÉTRICAS POR CLASE EN LOS OUTER FOLDS")
display(df_outer_comparativo[[
    "variante", "modelo", "outer_fold",
    "outer_f1_clase_0", "outer_f1_clase_1", "outer_f1_macro",
]].sort_values(["variante", "modelo", "outer_fold"]).round(4))

# Predicciones OOF: cada fila original debe tener una única predicción externa por modelo.
df_oof = pd.concat(OOF_REGISTROS, ignore_index=True)
resumen_oof = []
for (variante, modelo), grupo in df_oof.groupby(["variante", "modelo"]):
    _, y_ref = cargar_todos_preps(variante, split="train")
    assert len(grupo) == len(y_ref), f"OOF incompleto: {variante}/{modelo}"
    assert grupo["idx_train_oficial"].is_unique, f"OOF duplicado: {variante}/{modelo}"
    g = grupo.sort_values("idx_train_oficial")
    assert np.array_equal(g.y_true.to_numpy(), y_ref.to_numpy()), "Etiquetas OOF desalineadas"
    f1_c0, f1_c1 = f1_score(g.y_true, g.y_pred, labels=[0, 1], average=None, zero_division=0)
    resumen_oof.append({
        "variante": variante, "modelo": modelo,
        "f1_oof_global": f1_score(g.y_true, g.y_pred, average="macro"),
        "f1_oof_clase_0": f1_c0, "f1_oof_clase_1": f1_c1,
    })
df_oof_resumen = pd.DataFrame(resumen_oof)
print("\nRESUMEN OOF (métrica global distinta del promedio de los 5 folds)")
display(df_oof_resumen.sort_values(["variante", "modelo"]).round(4))

print("\nFRECUENCIA DEL PREPROCESAMIENTO GANADOR")
display(pd.crosstab(
    [df_outer_comparativo["variante"], df_outer_comparativo["modelo"]],
    df_outer_comparativo["best_preprocessing"],
))

### 6.1 Diagnóstico de hiperparámetros de XGBoost

Se informa qué proporción de los 5 Outer Folds seleccionó cada límite de los rangos explorados. Los límites repetidos son una **pista**, no una demostración, de que sería útil explorar otros intervalos. También se guarda cada trial para analizar la trayectoria de Optuna.

In [ ]:
LIMITES_XGB = {
    "n_estimators": (150, 700), "max_depth": (2, 7),
    "learning_rate": (0.01, 0.20), "min_child_weight": (2, 15),
    "subsample": (0.55, 0.90), "colsample_bytree": (0.50, 0.90),
    "gamma": (0.5, 8.0), "reg_alpha": (1e-3, 10.0),
    "reg_lambda": (0.1, 20.0), "scale_pos_weight": (1.0, 2.5),
}
filas_limites = []
for variante in VARIANTES:
    params_por_fold = nested_detalle[variante]["best_params"].tolist()
    for nombre, (lo, hi) in LIMITES_XGB.items():
        valores = [p[nombre] for p in params_por_fold]
        # Algunos parámetros log-uniformes rara vez coinciden con el límite exacto.
        amplitud = hi - lo
        cerca_bajo = sum(v <= lo + 0.02 * amplitud for v in valores)
        cerca_alto = sum(v >= hi - 0.02 * amplitud for v in valores)
        filas_limites.append({
            "variante": variante, "parametro": nombre,
            "min_rango": lo, "max_rango": hi,
            "cerca_min_5_folds": cerca_bajo, "cerca_max_5_folds": cerca_alto,
            "valores_elegidos": valores,
        })
df_diagnostico_limites = pd.DataFrame(filas_limites)
display(df_diagnostico_limites)

# ¿Cuántos de los 50 trials realmente evaluaron cada preprocessing?
conteos_trials = []
for variante in VARIANTES:
    for outer_fold, study in nested_studies[variante].items():
        for prep in PREPROCESAMIENTOS:
            t_ok = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE
                    and t.params.get("preprocessing") == prep]
            conteos_trials.append({"variante": variante, "outer_fold": outer_fold,
                                   "preprocessing": prep, "trials_completados": len(t_ok)})
df_conteos_trials = pd.DataFrame(conteos_trials)
print("\nDISTRIBUCIÓN DE TRIALS POR PREPROCESSING")
display(df_conteos_trials)

### 6.2 Guardar los resultados obtenidos **sin tocar test**

Los CSV de detalle y el historial de Optuna se guardan en un directorio exclusivo para este experimento. Los artefactos de tus notebooks originales permanecen intactos.

In [ ]:
df_nested_comparativo.to_csv(os.path.join(OUT_DIR, "resumen_nested_ml.csv"), index=False)
df_outer_comparativo.to_csv(os.path.join(OUT_DIR, "outer_folds_ml.csv"), index=False)
df_candidatos_inner_lr_rf.to_csv(os.path.join(OUT_DIR, "candidatos_inner_lr_rf.csv"), index=False)
df_diagnostico_limites.to_csv(os.path.join(OUT_DIR, "diagnostico_limites_xgb.csv"), index=False)
df_conteos_trials.to_csv(os.path.join(OUT_DIR, "conteos_trials_preprocessing_xgb.csv"), index=False)
df_oof.to_csv(os.path.join(OUT_DIR, "predicciones_oof_ml.csv"), index=False)
df_oof_resumen.to_csv(os.path.join(OUT_DIR, "resumen_oof_ml.csv"), index=False)
pd.DataFrame([VERSIONES]).to_csv(os.path.join(OUT_DIR, "versiones_paquetes.csv"), index=False)

for variante in VARIANTES:
    for outer_fold, study in nested_studies[variante].items():
        study.trials_dataframe().to_csv(
            os.path.join(OUT_DIR, f"optuna_xgb_{variante}_outer{outer_fold}.csv"),
            index=False,
        )

with open(os.path.join(OUT_DIR, "config_experimento.json"), "w", encoding="utf-8") as fh:
    json.dump({
        "random_state": RANDOM_STATE, "variantes": VARIANTES,
        "tfidf_comun": TFIDF_COMUN, "splitter": "StratifiedGroupKFold", "group_key": "transitive_normal_stem_lemma_message_exact",
        "group_audit": df_auditoria_grupos.to_dict(orient="records"),
        "n_outer": N_OUTER, "n_inner": N_INNER,
        "n_trials_nested_xgb": N_TRIALS_NESTED,
        "n_trials_final_xgb": N_TRIALS_FINAL,
        "grids_lr_rf": {m: grid_lr_rf(m) for m in MODELOS_LR_RF},
        "tfidf_config": {"ngram_range": [1, 2], "max_features": 20000,
                          "min_df": 2, "max_df": 0.98, "sublinear_tf": True,
                          "lowercase": False, "stop_words": None},
        "xgb_search": "TPE seeded; 3 initial preprocessing trials; total budget includes them",
        "versions": VERSIONES,
    }, fh, indent=2, default=str, ensure_ascii=False)
print(f"Resultados de desarrollo guardados en {OUT_DIR}")

## 7. Selección final — búsqueda independiente sobre todo el TRAIN

Después de terminar Nested CV, se ejecuta una **búsqueda nueva** en los 2400 registros de desarrollo: GridSearchCV para LR/RF y otro estudio Optuna TPE de 50 trials para XGBoost. El preprocessing se vuelve a seleccionar en CV; no se heredan los hiperparámetros ganadores de un Outer Fold. **Los nueve estimadores se ajustan y guardan con todo el train antes de abrir test.**

In [ ]:
FINAL_LR_RF = {}
filas_finales_lr_rf = []
for variante in VARIANTES:
    X_por_prep, y = cargar_todos_preps(variante, split="train")
    final_splits = crear_splits_final(y, grupos_train(variante), variante)
    for modelo in MODELOS_LR_RF:
        mejor, candidatos = buscar_lr_rf_train(X_por_prep, y, modelo, final_splits)
        FINAL_LR_RF[(variante, modelo)] = mejor
        filas_finales_lr_rf.append({
            "variante": variante, "modelo": modelo,
            "preprocessing": mejor["prep"],
            "best_f1_cv_final": mejor["inner_f1"],
            "cv_std_final": mejor["inner_std"],
            "best_params": mejor["params"],
        })

df_finales_lr_rf = pd.DataFrame(filas_finales_lr_rf)
print("SELECCIÓN FINAL LR/RF — TEST AÚN NO CARGADO")
display(df_finales_lr_rf)

In [ ]:
def optimizar_final_variante(variante, n_trials=N_TRIALS_FINAL):
    X_por_prep, y = cargar_todos_preps(variante, split="train")

    final_splits = crear_splits_final(y, grupos_train(variante), variante)

    def objective(trial):
        prep = trial.suggest_categorical(
            "preprocessing", list(PREPROCESAMIENTOS.keys())
        )
        params = sugerir_params_xgb(trial)
        pipeline = crear_pipeline(params, random_state=RANDOM_STATE)

        scores = cross_val_score(
            estimator=pipeline,
            X=X_por_prep[prep],
            y=y,
            cv=final_splits,
            scoring="f1_macro",
            n_jobs=1,
        )
        trial.set_user_attr("cv_f1_std", float(scores.std(ddof=1)))
        return float(scores.mean())

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    if n_trials >= len(PREPROCESAMIENTOS):
        for prep_inicial in PREPROCESAMIENTOS:
            study.enqueue_trial({"preprocessing": prep_inicial})
    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=True,
    )
    return study


studies_finales = {}

for variante in VARIANTES:
    print(f"\n{'='*70}")
    print(f"OPTIMIZACIÓN FINAL — {variante.upper()}")
    print(f"{'='*70}")

    study = optimizar_final_variante(
        variante, n_trials=N_TRIALS_FINAL
    )
    studies_finales[variante] = study

    print(f"Mejor F1-CV: {study.best_value:.4f}")
    print(f"Mejor preprocessing: {study.best_params['preprocessing']}")
    print("Mejores hiperparámetros XGBoost:")
    for k, v in study.best_params.items():
        if k != "preprocessing":
            print(f"  {k}: {repr(v)}")

In [ ]:
# Resumen de la selección final

filas = []
for variante, study in studies_finales.items():
    fila = {
        "variante": variante,
        "best_f1_cv_final": study.best_value,
        "cv_std_final": study.best_trial.user_attrs.get("cv_f1_std", np.nan),
    }
    fila.update(study.best_params)
    filas.append(fila)


df_params_finales = pd.DataFrame(filas)

display(
    df_params_finales.style.format({
        "best_f1_cv_final": "{:.4f}",
        "cv_std_final": "{:.4f}",
        "learning_rate": "{:.6f}",
        "subsample": "{:.4f}",
        "colsample_bytree": "{:.4f}",
        "gamma": "{:.4f}",
        "reg_alpha": "{:.6f}",
        "reg_lambda": "{:.6f}",
        "scale_pos_weight": "{:.4f}",
    })
)

### 7.1 Consolidar y guardar las selecciones finales

Los tres modelos han cerrado la búsqueda sobre todo el train. Esta celda guarda las configuraciones finales sin consultar el test oficial.

In [ ]:
filas_xgb_final = []
for variante, study in studies_finales.items():
    filas_xgb_final.append({
        "variante": variante, "modelo": "XGBoost",
        "preprocessing": study.best_params["preprocessing"],
        "best_f1_cv_final": study.best_value,
        "cv_std_final": study.best_trial.user_attrs.get("cv_f1_std", np.nan),
        "best_params": {k: v for k, v in study.best_params.items() if k != "preprocessing"},
    })
df_seleccion_final = pd.concat([
    df_finales_lr_rf, pd.DataFrame(filas_xgb_final),
], ignore_index=True).sort_values(["variante", "modelo"])
print("CONFIGURACIONES FINALES — NUEVE MODELOS, SIN TEST")
display(df_seleccion_final)
df_seleccion_final.to_csv(os.path.join(OUT_DIR, "seleccion_final_ml.csv"), index=False)

# Uniformización final: todos los estimadores se entrenan sobre TODO train sin abrir test.
MODELOS_FINALES = {}
for (variante, modelo), mejor in FINAL_LR_RF.items():
    estimador = mejor["estimator"]  # Ya está refit sobre train por GridSearchCV.
    MODELOS_FINALES[(variante, modelo)] = estimador
    joblib.dump(estimador, os.path.join(MODELS_DIR,
                f"{modelo.lower()}_{variante}_{mejor['prep']}.pkl"))
for variante, study in studies_finales.items():
    best_all = study.best_params.copy()
    prep = best_all.pop("preprocessing")
    X_por_prep, y = cargar_todos_preps(variante, split="train")
    estimador = crear_pipeline(best_all, random_state=RANDOM_STATE)
    estimador.fit(X_por_prep[prep], y)
    MODELOS_FINALES[(variante, "XGBoost")] = estimador
    joblib.dump(estimador, os.path.join(MODELS_DIR, f"xgboost_{variante}_{prep}.pkl"))
print("Modelos finales entrenados y guardados SOLO con train:", len(MODELOS_FINALES))
for variante, study in studies_finales.items():
    study.trials_dataframe().to_csv(
        os.path.join(OUT_DIR, f"optuna_xgb_final_{variante}.csv"),
        index=False,
    )

## 8. TEST OFICIAL (bloqueado por defecto)

Solo habilitar `EJECUTAR_TEST_OFICIAL=True` en la configuración **una vez cerrados** los hiperparámetros, TF-IDF, método de selección, análisis sobre development y modelos finales. La celda reutiliza los nueve modelos ya ajustados: no realiza ninguna optimización ni entrenamiento nuevo. La evaluación de test oficial debe hacerse una sola vez.

In [ ]:
RESULTADOS_TEST = []
PREDICCIONES_TEST = {}

if not EJECUTAR_TEST_OFICIAL:
    print("TEST OFICIAL BLOQUEADO: ya se ajustaron y guardaron todos los modelos sin abrir test.")
else:
    for _, fila in df_seleccion_final.sort_values(["variante", "modelo"]).iterrows():
        variante, modelo, prep = fila["variante"], fila["modelo"], fila["preprocessing"]
        estimador = MODELOS_FINALES[(variante, modelo)]
        # Test se abre solo aquí, después de la selección y el fit final.
        df_test = cargar_split(variante, prep, split="test")
        X_test, y_test = preparar_xy(df_test)
        pred = estimador.predict(X_test)
        ref_cv = df_nested_comparativo.query(
            "variante == @variante and modelo == @modelo"
        ).iloc[0]
        # Métrica de sensibilidad: se muestra junto al score oficial, sin escoger nada con test.
        if variante not in SOLAPE_TEST_SIN_ETIQUETAS:
            raise ValueError("Falta auditoría de integridad del test: verificar CSV originales")
        mascara_no_solape = ~SOLAPE_TEST_SIN_ETIQUETAS[variante]
        if len(mascara_no_solape) != len(y_test):
            raise AssertionError("Alineación del test entre preprocesamientos incorrecta")
        f1_test_sin_solape = (f1_score(y_test.iloc[mascara_no_solape],
                                pred[mascara_no_solape], average="macro", zero_division=0)
                              if np.any(mascara_no_solape) else np.nan)
        RESULTADOS_TEST.append({
            "variante": variante, "modelo": modelo, "preprocessing": prep,
            "outer_f1_mean": ref_cv["f1_nested_mean"],
            "outer_f1_std": ref_cv["f1_nested_std"],
            "f1_cv_final": fila["best_f1_cv_final"],
            "f1_macro_test": f1_score(y_test, pred, average="macro"),
            "n_test_con_solape_normalizado": int((~mascara_no_solape).sum()),
            "n_test_sin_solape": int(mascara_no_solape.sum()),
            "f1_macro_test_sin_solape_sensibilidad": f1_test_sin_solape,
            "accuracy_test": accuracy_score(y_test, pred),
            "precision_macro_test": precision_score(y_test, pred, average="macro", zero_division=0),
            "recall_macro_test": recall_score(y_test, pred, average="macro", zero_division=0),
            "f1_clase_0": f1_score(y_test, pred, pos_label=0, zero_division=0),
            "f1_clase_1": f1_score(y_test, pred, pos_label=1, zero_division=0),
        })
        PREDICCIONES_TEST[(variante, modelo)] = (y_test, pred, df_test)
        print(f"{variante.upper()} | {modelo} | {prep}: TEST F1-Macro="
              f"{RESULTADOS_TEST[-1]['f1_macro_test']:.4f}")
    df_resultados_test = pd.DataFrame(RESULTADOS_TEST)
    display(df_resultados_test.sort_values(["variante", "modelo"]).round(4))
    df_resultados_test.to_csv(os.path.join(OUT_DIR, "resultados_test_ml.csv"), index=False)
    print("EVALUACIÓN FINAL TERMINADA: no volver a seleccionar hiperparámetros en función del test.")


### 8.1 Análisis de errores en los nueve modelos (solo cuando test esté habilitado)

El reporte por clase y la matriz de confusión se presentan para cada variante y modelo. No deben utilizarse para seguir modificando el procedimiento y escoger una nueva versión que optimice este mismo test.

In [ ]:
if not EJECUTAR_TEST_OFICIAL:
    print("Análisis de errores no disponible: test oficial bloqueado.")
else:
    for (variante, modelo), (y_test, pred, df_test) in PREDICCIONES_TEST.items():
        print(f"\n{'='*60}\n{variante.upper()} / {modelo}")
        print(classification_report(y_test, pred, digits=4, zero_division=0))
        print("Matriz de confusión:")
        print(confusion_matrix(y_test, pred))

### 8.2 Contribución de las nueve características lingüísticas (exploratorio)

Los nueve modelos finales ya están ajustados con el train oficial, aun con el test deshabilitado. Por ello es posible estudiar sus atributos sin consultar el test. **No compares directamente la magnitud de un coeficiente de LR con la importancia de RF/XGBoost**: son medidas de naturaleza distinta. No utilices estos resultados para reajustar un modelo y luego presentar el mismo test como independiente si ya lo evaluaste.

In [ ]:
def importancia_linguistica(estimador, modelo):
    paso_clf = "xgb" if modelo == "XGBoost" else "clf"
    clf = estimador.named_steps[paso_clf]
    if hasattr(clf, "feature_importances_"):
        importancia = np.asarray(clf.feature_importances_)
    elif hasattr(clf, "coef_"):
        importancia = np.abs(clf.coef_[0])
    else:
        return None
    preproc = estimador.named_steps["features"]
    n_tfidf = len(preproc.named_transformers_["tfidf"].get_feature_names_out())
    ling = importancia[n_tfidf:n_tfidf + len(FEATURE_COLS)]
    if len(ling) != len(FEATURE_COLS):
        return None
    return dict(zip(FEATURE_COLS, ling))

filas = []
for (variante, modelo), estimador in MODELOS_FINALES.items():
    imp = importancia_linguistica(estimador, modelo)
    if imp is not None:
        for nombre, valor in imp.items():
            filas.append({
                "variante": variante, "modelo": modelo,
                "caracteristica": nombre, "importancia": valor,
            })
df_importancias_linguisticas = pd.DataFrame(filas)
display(df_importancias_linguisticas)
df_importancias_linguisticas.to_csv(
    os.path.join(OUT_DIR, "importancias_linguisticas_ml.csv"), index=False
)

## Lectura e interpretación de esta corrida agrupada

1. Los Outer folds están **agrupados**; no se comparte entre train/validación ningún grupo conectado por coincidencia textual en normal, stem, lemma o mensaje original exacto. La auditoría guarda las comprobaciones para los 5 Outer × 5 Inner, además de la búsqueda final.
2. `F1 Train`, `F1 Outer`, resultados OOF y F1 por clase se calculan como antes. Las variaciones respecto de la corrida no agrupada pueden deberse tanto al cambio de particiones como a la representación textual.
3. Contradicción conocida en ES: los registros están juntos en un grupo; **no** se cambia la etiqueta original automáticamente. La falta de 3 filas MX / 4 ES sigue pendiente de verificar contra el corpus de origen.
4. El test oficial permanece bloqueado hasta que elijas y documentes el procedimiento de evaluación. La auditoría del test lee texto/ID, **nunca** sus etiquetas; si hay solapamiento normalizado, el informe final puede mostrar por separado el F1 oficial completo y el F1 de sensibilidad en el subconjunto sin coincidencias, pero nunca elegir hiperparámetros mediante estos resultados.
5. **Comparación entre los dos notebooks de Group CV:** mismos grupos, seeds, versiones de datos, clasificadores, 50 trials y características lingüísticas; cambia solo el **perfil TF-IDF** (y la selección de n-gramas interna propia del perfil LR/RF).
